# IMVCDC Multistage Refactor - Colab T4 Training Notebook

This notebook is prepared for Google Colab with a T4 GPU.

## What it does
1. Verifies GPU and CUDA.
2. Clones your project structure (IMVCDC + datasets) in DCG style.
3. Installs required Python packages.
4. Checks shared dataset files under `datasets/`.
5. Runs smoke test and stage training commands.

Set Runtime -> Change runtime type -> GPU before running.

In [ ]:
import torch
import subprocess

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
subprocess.run(['nvidia-smi'])

## 1) Repository Setup (DCG Style)

Choose one option:
- Option A: Download from GitHub ZIP and extract only IMVCDC + datasets
- Option B: Use Google Drive path directly

In [ ]:
import os
import subprocess

BASE_DIR = '/content'

# OPTION A: Download from GitHub (ZIP)
USE_GITHUB = True

if USE_GITHUB:
    PROJECT_ROOT = os.path.join(BASE_DIR, 'Diffusion-based-approaches')
    ZIP_PATH = os.path.join(BASE_DIR, 'repo.zip')
    EXTRACTED_FOLDER = os.path.join(BASE_DIR, 'Diffusion-based-approaches-for-IMVC-main')
    IMVCDC_SRC = os.path.join(EXTRACTED_FOLDER, 'IMVCDC')
    DATASETS_SRC = os.path.join(EXTRACTED_FOLDER, 'datasets')
    IMVCDC_DIR = os.path.join(PROJECT_ROOT, 'IMVCDC')
    DATASETS_DIR = os.path.join(PROJECT_ROOT, 'datasets')

    os.makedirs(PROJECT_ROOT, exist_ok=True)

    # Download ZIP
    subprocess.run([
        'wget', '-q', '-O', ZIP_PATH,
        'https://github.com/EddarirOmar/Diffusion-based-approaches-for-IMVC/archive/refs/heads/main.zip'
    ], check=True)

    # Extract IMVCDC and datasets folders only
    subprocess.run([
        'unzip', '-q', ZIP_PATH,
        'Diffusion-based-approaches-for-IMVC-main/IMVCDC/*',
        'Diffusion-based-approaches-for-IMVC-main/datasets/*'
    ], check=True)

    # Refresh destination folders
    if os.path.exists(IMVCDC_DIR):
        subprocess.run(['rm', '-rf', IMVCDC_DIR], check=True)
    if os.path.exists(DATASETS_DIR):
        subprocess.run(['rm', '-rf', DATASETS_DIR], check=True)

    subprocess.run(['mv', IMVCDC_SRC, PROJECT_ROOT], check=True)
    subprocess.run(['mv', DATASETS_SRC, PROJECT_ROOT], check=True)

# OPTION B: Use Google Drive
else:
    PROJECT_ROOT = '/content/drive/MyDrive/path-to-your-repo/Diffusion-based-approaches'
    IMVCDC_DIR = os.path.join(PROJECT_ROOT, 'IMVCDC')
    DATASETS_DIR = os.path.join(PROJECT_ROOT, 'datasets')

print('PROJECT_ROOT:', PROJECT_ROOT)
print('IMVCDC_DIR:', IMVCDC_DIR)
print('DATASETS_DIR:', DATASETS_DIR)
print('IMVCDC exists:', os.path.exists(IMVCDC_DIR))
print('datasets exists:', os.path.exists(DATASETS_DIR))

## 2) Install Dependencies

In [ ]:
# Install required dependencies for IMVCDC
%pip install -q numpy scipy scikit-learn pyyaml tqdm matplotlib munkres

In [ ]:
!python -m pip install --upgrade pip
!pip install numpy scipy scikit-learn pyyaml tqdm matplotlib munkres
# Install torch only if needed in your Colab runtime.
# !pip install torch torchvision torchaudio

## 3) Quick Dataset Presence Check (shared datasets folder)

In [ ]:
import os

data_dir = DATASETS_DIR
required = [
    'synthetic3d.mat',
    'NoisyMNIST.mat',
    'cub_googlenet_doc2vec_c10.mat',
]

print('Data dir:', data_dir)
for f in required:
    p = os.path.join(data_dir, f)
    print(f, '->', 'FOUND' if os.path.exists(p) else 'MISSING')

## 4) Optional Quick Sanity Check

In [ ]:
%cd {IMVCDC_DIR}
!python smoke_test.py

## 5) Final Training (Stage-based)

Pick a dataset and run stages in order.

In [ ]:
# Dataset names supported by configure.py include:
# Synthetic3d, CUB, NoisyMNIST, NoisyDigit-Product, Multi-Fashion, Multi-Coil20, HandWritten, LandUse-21, Scene-15, Caltech101-7

DATASET_NAME = 'Synthetic3d'
%cd {IMVCDC_DIR}
!python run_stage1.py --data_name {DATASET_NAME}

## 6) Stage 2

In [ ]:
%cd {IMVCDC_DIR}
!python run_stage2.py --data_name {DATASET_NAME}

## 7) Stage 3

Stage 3 requires Stage 2 completed latents (`test_latents_completed.npz`).

In [ ]:
from pathlib import Path

slug = DATASET_NAME.lower().replace('-', '').replace(' ', '')
required_file = Path('outputs') / f'stage2_dm_{slug}' / 'latents' / 'test_latents_completed.npz'
print('Expected Stage 2 output:', required_file)

%cd {IMVCDC_DIR}
if required_file.exists():
    print('Stage 2 output found. Running Stage 3...')
    !python run_stage3.py --data_name {DATASET_NAME}
else:
    print('Stage 3 skipped: completed latent file not found.')
    print('Please ensure run_stage2.py generates test_latents_completed.npz for your current branch.')

## 8) Optional Unified Runner

Use this if you prefer one entry command (wrapper around stage scripts).

In [ ]:
%cd {IMVCDC_DIR}
# !python run.py --stage 1 --data_name {DATASET_NAME}
# !python run.py --stage all --data_name {DATASET_NAME}